# Verification of reported feature-importance numbers

Recomputes every number in `window_FI_report_EN.md` from the raw per-seed
matrices in `fi_per_seed.npz`, and repairs the all-NaN `shap` column in
`fi_table_TS.csv` / `fi_table_late.csv`.

Run top to bottom. Nothing is overwritten in place — repaired tables are
written as `*_fixed.csv`.

**If Cell 2 reports missing arrays, paste its output and stop.**


In [1]:
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, fisher_exact

NPZ     = "fi_per_seed.npz"
NAMES   = "feature_names.csv"
TOP_K   = 15
TOP_N   = 50
WINDOWS = ["TS", "late"]
METHODS = ["gini", "perm", "shap"]

z = np.load(NPZ, allow_pickle=True)
print("Arrays in", NPZ)
for k in z.files:
    print(f"  {k:<28} shape={str(z[k].shape):<16} dtype={z[k].dtype}")

Arrays in fi_per_seed.npz
  TS__gini                     shape=(10, 274)        dtype=float64
  TS__perm                     shape=(10, 274)        dtype=float64
  TS__shap                     shape=(10, 274)        dtype=float64
  late__gini                   shape=(10, 274)        dtype=float64
  late__perm                   shape=(10, 274)        dtype=float64
  late__shap                   shape=(10, 274)        dtype=float64
  seeds                        shape=(10,)            dtype=int64
  feature_names                shape=(274,)           dtype=<U12
  feature_kind                 shape=(274,)           dtype=<U7
  feature_resid                shape=(274,)           dtype=int64
  TS__oof                      shape=(10,)            dtype=float64
  TS__window                   shape=(2,)             dtype=int64
  late__oof                    shape=(10,)            dtype=float64
  late__window                 shape=(2,)             dtype=int64


## 1 — Locate the arrays

Keys are matched by containing both a window name and a method name. If your
naming differs, edit `find()` using the listing above.

In [2]:
def find(window, method):
    c = [k for k in z.files
         if window.lower() in k.lower() and method.lower() in k.lower()]
    if len(c) == 1:
        return z[c[0]]
    if len(c) > 1:
        exact = [k for k in c if k.lower() in (f"{window}_{method}".lower(),
                                               f"{method}_{window}".lower())]
        print(f"  ambiguous {window}/{method}: {c} -> using {(exact or c)[0]}")
        return z[(exact or c)[0]]
    return None

M, missing = {}, []
for w in WINDOWS:
    for m in METHODS:
        a = find(w, m)
        (M.__setitem__((w, m), np.asarray(a, float)) if a is not None
         else missing.append(f"{w}/{m}"))

if missing:
    print("MISSING:", missing, "-> paste the listing above and stop here")

n_seeds, n_feat = next(iter(M.values())).shape
names    = pd.read_csv(NAMES)
is_water = (names["kind"].astype(str).str.lower() == "water").to_numpy()
name_arr = names["name"].astype(str).to_numpy()
assert len(is_water) == n_feat, f"{len(is_water)} names vs {n_feat} features"

print(f"\n{len(M)} arrays | n_seeds={n_seeds} | n_features={n_feat} "
      f"| waters={is_water.sum()} (baseline {is_water.mean():.1%})")


6 arrays | n_seeds=10 | n_features=274 | waters=50 (baseline 18.2%)


## 2 — Helpers

`split_half` uses all C(n,n/2)/2 distinct partitions and reports medians, matching
the procedure recorded in `reproducibility.json` (126 partitions for 10 seeds).
Jaccard is intersection over union, so a value `j` corresponds to
`30j/(1+j)` shared features at k = 15.

In [3]:
def topk(v, k=TOP_K):
    return set(np.argsort(-np.nan_to_num(v, nan=-np.inf))[:k])

def jac(a, b, k=TOP_K):
    A, B = topk(a, k), topk(b, k)
    return len(A & B) / len(A | B)

def shared(j, k=TOP_K):
    return round(2 * k * j / (1 + j))

PARTS = [set(c) for c in combinations(range(n_seeds), n_seeds // 2)]
PARTS = PARTS[: len(PARTS) // 2]

def split_half(mat):
    rr, jj = [], []
    for A in PARTS:
        B = [i for i in range(n_seeds) if i not in A]
        a, b = mat[sorted(A)].mean(0), mat[B].mean(0)
        rr.append(spearmanr(a, b).statistic)
        jj.append(jac(a, b))
    return float(np.median(rr)), float(np.median(jj))

print(f"{len(PARTS)} split-half partitions")

126 split-half partitions


## 3 — Reproducibility floor

Report claims: Gini 0.976/0.977 & J 0.765/0.875 · perm 0.116/0.128 & J 0.200/0.034
· SHAP 0.964/0.977 & J 0.875/1.000.

In [4]:
rows = []
for m in METHODS:
    for w in WINDOWS:
        if (w, m) in M:
            r, j = split_half(M[(w, m)])
            rows.append({"method": m, "window": w, "floor_rho": round(r, 3),
                         "floor_J": round(j, 3), "shared_of_15": shared(j)})
floors = pd.DataFrame(rows)
display(floors)

,method,window,floor_rho,floor_J,shared_of_15
0,gini,TS,0.976,0.765,13
1,gini,late,0.977,0.875,14
2,perm,TS,0.116,0.200,5
3,perm,late,0.128,0.034,1
4,shap,TS,0.964,0.875,14
5,shap,late,0.977,1.000,15


## 4 — Cross-window agreement

Report claims: Gini +0.398 · perm −0.107 · SHAP +0.375, disattenuated ≈ 0.39.

Disattenuation is skipped where either floor is below 0.5 — dividing a near-zero
correlation by the root of two near-zero reliabilities returns a confident
but meaningless number.

In [5]:
rows = []
for m in METHODS:
    if ("TS", m) in M and ("late", m) in M:
        a, b = M[("TS", m)].mean(0), M[("late", m)].mean(0)
        rho, j = spearmanr(a, b).statistic, jac(a, b)
        rts = floors.query("method==@m and window=='TS'").floor_rho.iloc[0]
        rla = floors.query("method==@m and window=='late'").floor_rho.iloc[0]
        dis = rho / np.sqrt(rts * rla) if min(rts, rla) > 0.5 else np.nan
        rows.append({"method": m, "cross_rho": round(rho, 3), "cross_J": round(j, 3),
                     "shared_of_15": shared(j), "floor_TS": rts, "floor_late": rla,
                     "disattenuated": None if np.isnan(dis) else round(dis, 3)})
display(pd.DataFrame(rows))

,method,cross_rho,cross_J,shared_of_15,floor_TS,floor_late,disattenuated
0,gini,0.398,0.2,5,0.976,0.977,0.407
1,perm,-0.107,0.0,0,0.116,0.128,NaN
2,shap,0.375,0.2,5,0.964,0.977,0.387


## 5 — Method agreement within each window

Figure 1 shows: TS 0.11 / 0.97 / 0.12 and late −0.23 / 0.98 / −0.23.

In [6]:
rows = []
for w in WINDOWS:
    for m1, m2 in [("gini", "perm"), ("gini", "shap"), ("perm", "shap")]:
        if (w, m1) in M and (w, m2) in M:
            a, b = M[(w, m1)].mean(0), M[(w, m2)].mean(0)
            rows.append({"window": w, "pair": f"{m1}-{m2}",
                         "rho": round(spearmanr(a, b).statistic, 3),
                         "J": round(jac(a, b), 3)})
display(pd.DataFrame(rows))

,window,pair,rho,J
0,TS,gini-perm,0.108,0.154
1,TS,gini-shap,0.972,0.875
2,TS,perm-shap,0.124,0.154
3,late,gini-perm,-0.230,0.071
4,late,gini-shap,0.975,0.875
5,late,perm-shap,-0.233,0.071


## 6 — Distribution of permutation values

The report states that 42.5% of late-window permutation importances are *exactly*
zero (13.2% in TS). The seed-averaged tables show almost no exact zeros, so the
claim can only refer to the per-seed values before averaging. This cell reports
both layers so the correct statement can be used.

In [7]:
rows = []
for w in WINDOWS:
    if (w, "perm") in M:
        raw, avg = M[(w, "perm")].ravel(), M[(w, "perm")].mean(0)
        for layer, v in [("per-seed", raw), ("seed-averaged", avg)]:
            rows.append({"window": w, "layer": layer, "n": v.size,
                         "exactly_0": f"{(v == 0).mean():.1%}",
                         "non_positive": f"{(v <= 0).mean():.1%}",
                         "positive": f"{(v > 0).mean():.1%}"})
display(pd.DataFrame(rows))

,window,layer,n,exactly_0,non_positive,positive
0,TS,per-seed,2740,12.9%,62.2%,37.8%
1,TS,seed-averaged,274,0.0%,69.0%,31.0%
2,late,per-seed,2740,42.4%,85.8%,14.2%
3,late,seed-averaged,274,0.4%,90.5%,9.5%


## 7 — Water enrichment in the top 50

Report claims: Gini 11 vs 1 (OR 13.8) · perm 14 vs 7 (OR 2.4) · SHAP 13 vs 2
(OR 8.4). Baseline is 50/274 = 18.2%.

In [8]:
rows = []
for m in METHODS:
    if ("TS", m) in M and ("late", m) in M:
        c = {w: int(is_water[np.argsort(-np.nan_to_num(M[(w, m)].mean(0),
                                                       nan=-np.inf))[:TOP_N]].sum())
             for w in WINDOWS}
        orr, p = fisher_exact([[c["TS"], TOP_N - c["TS"]],
                               [c["late"], TOP_N - c["late"]]])
        rows.append({"method": m, "TS": f"{c['TS']}/{TOP_N}",
                     "TS_pct": f"{c['TS']/TOP_N:.0%}", "late": f"{c['late']}/{TOP_N}",
                     "late_pct": f"{c['late']/TOP_N:.0%}",
                     "odds_ratio": round(orr, 2), "fisher_p": round(p, 4)})
display(pd.DataFrame(rows))

,method,TS,TS_pct,late,late_pct,odds_ratio,fisher_p
0,gini,11/50,22%,1/50,2%,13.82,0.0038
1,perm,14/50,28%,7/50,14%,2.39,0.1396
2,shap,13/50,26%,2/50,4%,8.43,0.0038


## 8 — Leading features

In [9]:
for m in METHODS:
    for w in WINDOWS:
        if (w, m) in M:
            idx = np.argsort(-np.nan_to_num(M[(w, m)].mean(0), nan=-np.inf))[:10]
            print(f"{m+' '+w:<12} {list(name_arr[idx])}")

gini TS      ['SER145', 'CYS59', 'GLY214', 'TYR60', 'SER62', 'LYS87', 'PHE42', 'TYR94', 'CYS43', 'GLY146']
gini late    ['GLY214', 'ASN219', 'SER144', 'LYS220', 'SER145', 'TYR94', 'LYS143', 'TYR60', 'ALA169', 'SER62']
perm TS      ['water rank28', 'water rank41', 'water rank3', 'water rank2', 'GLY63', 'water rank43', 'water rank24', 'SER145', 'ILE174', 'water rank6']
perm late    ['TYR183', 'TYR32', 'LEU184', 'water rank39', 'VAL199', 'ASN49', 'VAL209', 'LYS220', 'water rank36', 'LYS226']
shap TS      ['SER145', 'CYS59', 'PHE42', 'TYR60', 'TYR94', 'SER62', 'GLY146', 'LYS87', 'ILE64', 'CYS43']
shap late    ['GLY214', 'SER144', 'ASN219', 'SER145', 'LYS220', 'TYR94', 'TYR60', 'LYS143', 'ALA169', 'SER62']


## 9 — Repair the exported tables

`fi_table_TS.csv` and `fi_table_late.csv` currently have an all-NaN `shap`
column, because SHAP was computed in a later backfill that never wrote back to
them. This rewrites all three importance columns from the per-seed arrays.

In [10]:
for w, fn in [("TS", "fi_table_TS.csv"), ("late", "fi_table_late.csv")]:
    df = pd.read_csv(fn)
    before = int(df["shap"].isna().sum()) if "shap" in df else -1
    for m in METHODS:
        if (w, m) in M:
            df[m] = M[(w, m)].mean(0)
    out = fn.replace(".csv", "_fixed.csv")
    df.to_csv(out, index=False)
    print(f"{fn}: shap NaN {before} -> {int(df['shap'].isna().sum())}  ->  {out}")

fi_table_TS.csv: shap NaN 274 -> 0  ->  fi_table_TS_fixed.csv
fi_table_late.csv: shap NaN 274 -> 0  ->  fi_table_late_fixed.csv


## 10 — Redraw Figure 4.3

The original scatter has its title overlapping the y-axis label. This version
puts the title on two lines and sizes axis labels to sit alongside 12 pt body
text. Uses the repaired SHAP values.

In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"axes.labelsize": 13, "axes.titlesize": 13,
                     "xtick.labelsize": 12, "ytick.labelsize": 12,
                     "legend.fontsize": 11})

x = np.nan_to_num(M[("TS", "shap")].mean(0));   x = x / x.max()
y = np.nan_to_num(M[("late", "shap")].mean(0)); y = y / y.max()

cross = pd.DataFrame(rows) if False else None
rho_x = spearmanr(x, y).statistic
j_x   = jac(x, y)
f_ts  = floors.query("method=='shap' and window=='TS'").iloc[0]
f_la  = floors.query("method=='shap' and window=='late'").iloc[0]

fig, ax = plt.subplots(figsize=(7.2, 6.4))
ax.plot([0, 1.02], [0, 1.02], ls="--", lw=1.2, color="0.55", label="y = x", zorder=1)
ax.scatter(x[~is_water], y[~is_water], s=42, c="#1f77b4", alpha=.85,
           edgecolors="none", label="protein residue", zorder=2)
ax.scatter(x[is_water], y[is_water], s=42, c="#f0a830", alpha=.9,
           edgecolors="none", label="water (rank-based)", zorder=3)

for i in np.argsort(-np.maximum(x, y))[:8]:
    dx, dy = (0.018, 0.012) if x[i] >= y[i] else (-0.018, 0.016)
    ax.annotate(name_arr[i], (x[i], y[i]), xytext=(x[i]+dx, y[i]+dy),
                fontsize=10.5, fontweight="bold", color="#12507f",
                ha="left" if dx > 0 else "right", zorder=4)

ax.set_xlabel("TS window importance  (frames 500\u20131500, normalised)")
ax.set_ylabel("Late window importance  (frames 2000\u20132499, normalised)")
ax.set_title("SHAP importance: transition vs stable-approach\n"
             f"cross-window \u03c1 = {rho_x:+.3f}    within-window floor "
             f"\u03c1: TS {f_ts.floor_rho:+.3f}, late {f_la.floor_rho:+.3f}",
             pad=16, linespacing=1.5)
ax.text(.025, .975, f"top-15 Jaccard\ncross-window {j_x:.3f}\n"
        f"floor: TS {f_ts.floor_J:.3f} / late {f_la.floor_J:.3f}",
        transform=ax.transAxes, va="top", fontsize=10.5,
        bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.6", lw=.9), zorder=5)
ax.legend(loc="upper right", frameon=False)
ax.set_xlim(-.02, 1.10); ax.set_ylim(-.02, 1.10)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("fig43_scatter.png", dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig("fig43_scatter.pdf", bbox_inches="tight", facecolor="white")
print(f"saved fig43_scatter.png / .pdf   (cross rho={rho_x:+.3f}, J={j_x:.3f})")

saved fig43_scatter.png / .pdf   (cross rho=+0.375, J=0.200)
